In [96]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bryanpark/sudoku")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\bryanpark\sudoku\versions\3


In [97]:
import copy
import sudoku
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


solution = sudoku.construct_puzzle_solution()
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)
print(puzzle)
sudoku.display(puzzle)
print("givens:", givens)

[[0, 8, 7, 3, 2, 0, 0, 6, 0], [6, 0, 9, 0, 1, 0, 8, 0, 4], [0, 2, 1, 0, 0, 9, 0, 0, 5], [0, 9, 0, 6, 0, 0, 0, 8, 1], [2, 0, 0, 4, 0, 0, 9, 0, 3], [0, 5, 4, 0, 8, 0, 0, 7, 0], [3, 1, 0, 0, 4, 0, 0, 0, 0], [0, 0, 0, 7, 9, 6, 0, 1, 2], [0, 0, 2, 0, 0, 5, 6, 0, 8]]
_ 8 7 3 2 _ _ 6 _
6 _ 9 _ 1 _ 8 _ 4
_ 2 1 _ _ 9 _ _ 5
_ 9 _ 6 _ _ _ 8 1
2 _ _ 4 _ _ 9 _ 3
_ 5 4 _ 8 _ _ 7 _
3 1 _ _ 4 _ _ _ _
_ _ _ 7 9 6 _ 1 2
_ _ 2 _ _ 5 6 _ 8
givens: 38


In [98]:
import numpy as np
quizzes = np.zeros((1000000, 81), np.int32)
solutions = np.zeros((1000000, 81), np.int32)
for i, line in enumerate(open('data/sudoku.csv', 'r').read().splitlines()[1:]):
    quiz, solution = line.split(",")
    for j, q_s in enumerate(zip(quiz, solution)):
        q, s = q_s
        quizzes[i, j] = q
        solutions[i, j] = s
quizzes = quizzes.reshape((-1, 9, 9))
solutions = solutions.reshape((-1, 9, 9))

In [99]:
print(quizzes)

[[[0 0 4 ... 2 0 9]
  [0 0 5 ... 0 0 1]
  [0 7 0 ... 0 4 3]
  ...
  [6 0 0 ... 1 0 5]
  [0 0 3 ... 6 9 0]
  [0 4 2 ... 3 0 0]]

 [[0 4 0 ... 0 5 0]
  [1 0 7 ... 9 6 0]
  [5 2 0 ... 0 0 0]
  ...
  [0 9 0 ... 5 4 3]
  [6 0 0 ... 7 0 0]
  [2 5 0 ... 1 0 0]]

 [[6 0 0 ... 3 8 4]
  [0 0 8 ... 0 7 2]
  [0 0 0 ... 0 0 5]
  ...
  [3 1 0 ... 0 5 0]
  [0 8 9 ... 0 0 0]
  [5 0 2 ... 1 9 0]]

 ...

 [[0 0 0 ... 8 2 0]
  [0 6 1 ... 0 3 0]
  [0 5 0 ... 0 0 0]
  ...
  [0 0 7 ... 0 6 5]
  [0 0 0 ... 4 0 8]
  [0 8 6 ... 0 0 0]]

 [[0 7 0 ... 6 9 0]
  [0 0 3 ... 0 0 1]
  [0 0 0 ... 0 2 0]
  ...
  [0 0 0 ... 0 4 0]
  [0 5 1 ... 9 0 0]
  [9 4 0 ... 0 0 7]]

 [[3 0 0 ... 6 2 0]
  [1 0 0 ... 4 0 0]
  [0 0 5 ... 8 3 0]
  ...
  [4 8 0 ... 0 1 0]
  [2 0 3 ... 0 0 0]
  [0 7 0 ... 0 9 0]]]


In [100]:
df = pd.read_csv('data/sudoku.csv',dtype=str)
df

,quizzes,solutions
0,0043002090050090010700600430060020871900074000...,8643712593258497619712658434361925871986574322...
1,0401000501070039605200080000000000170009068008...,3461792581875239645296483719658324174729168358...
2,6001203840084590720000060050002640300700800069...,6951273841384596727248369158512647392739815469...
3,4972000001004000050000160986203000403009000000...,4972583161864397252537164986293815473759641828...
4,0059103080094030600275001000300002010008200070...,4659123781894735623275681497386452919548216372...
...,...,...
999995,3000280000290000300054001077402030980086070031...,3175289464291768356854391277462135989586472131...
999996,0030006000040860057000009409350407208067200502...,5234976811942863757685139429356417288167294532...
999997,0003508200618040300500090000700600029030070100...,7493568212618745393582197468749613529235876146...
999998,0702006900030400010000650205600300000947005800...,4752816936239478511893657245628341793947165828...


In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   quizzes    1000000 non-null  object
 1   solutions  1000000 non-null  object
dtypes: object(2)
memory usage: 15.3+ MB


In [102]:
# 81자리가 아닌 문제 확인 > 없음
df[df['quizzes'].str.len()!=81]

,quizzes,solutions


In [103]:
# 81자리가 아닌 답안지 확인 > 없음
df[df['solutions'].str.len()!=81]

,quizzes,solutions


입력(quiz) :  0~9 (0은 빈칸을 뜻함)  
정답(solution) : 1~9  
학습 라벨 : (solution - 1) → 0~8

In [104]:
quiz_str = df['quizzes'][0]
quiz = np.array([int(c) for c in quiz_str]).reshape(9,9).astype(np.int64)
quiz

array([[0, 0, 4, 3, 0, 0, 2, 0, 9],
       [0, 0, 5, 0, 0, 9, 0, 0, 1],
       [0, 7, 0, 0, 6, 0, 0, 4, 3],
       [0, 0, 6, 0, 0, 2, 0, 8, 7],
       [1, 9, 0, 0, 0, 7, 4, 0, 0],
       [0, 5, 0, 0, 8, 3, 0, 0, 0],
       [6, 0, 0, 0, 0, 0, 1, 0, 5],
       [0, 0, 3, 5, 0, 8, 6, 9, 0],
       [0, 4, 2, 9, 1, 0, 3, 0, 0]])

In [105]:


class SudokuDataset(Dataset):
    """
    반환:
      X: (10, 9, 9) float32 one-hot (0~9)
      y: (9, 9) int64  (0~8)  # 정답 1~9를 0~8로 shift
      mask: (9, 9) bool  # quiz==0 (빈칸 위치)
    """
    def __init__(self, df: pd.DataFrame, quiz_col: str, sol_col: str):
        self.quiz = df[quiz_col].values
        self.sol  = df[sol_col].values

    def __len__(self):
        return len(self.quiz)

    @staticmethod
    def _str_to_grid(s: str) -> np.ndarray:

        b = np.frombuffer(s.encode("ascii"), dtype=np.uint8) - ord("0")
        return b.reshape(9, 9).astype(np.int64)

    def __getitem__(self, idx: int):
        quiz_str = self.quiz[idx]
        sol_str  = self.sol[idx]

        quiz = self._str_to_grid(quiz_str)     # (9,9) 0~9
        sol  = self._str_to_grid(sol_str)      # (9,9) 1~9

        mask = (quiz == 0)                                  # 마스크: 빈칸 위치만 학습,평가에 쓰는 용도

        # 라벨: 1~9 -> 0~8
        y = sol - 1

        # 입력 원핫: 0~9 (10클래스)
        # X[d, i, j] = 1 if quiz[i,j] == d
        X = np.zeros((10, 9, 9), dtype=np.float32)
        for d in range(10):
            X[d] = (quiz == d)

        # torch 텐서로 변환
        X = torch.from_numpy(X)                 # float32
        y = torch.from_numpy(y.astype(np.int64))# int64
        mask = torch.from_numpy(mask)           # bool

        return X, y, mask

In [106]:
dataset = SudokuDataset(df, "quizzes", "solutions")

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,      # 윈도우면 0~2부터 시작 추천
    pin_memory=True
)

X, y, mask = next(iter(loader))
print("X:", X.shape, X.dtype)           # (B, 10, 9, 9) torch.float32
print("y:", y.shape, y.dtype)           # (B, 9, 9) torch.int64
print("mask:", mask.shape, mask.dtype)  # (B, 9, 9) torch.bool
print("blank ratio in batch:", mask.float().mean().item())

X: torch.Size([64, 10, 9, 9]) torch.float32
y: torch.Size([64, 9, 9]) torch.int64
mask: torch.Size([64, 9, 9]) torch.bool
blank ratio in batch: 0.5817901492118835


c:\Users\Playdata\deep_learning\dl_venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [107]:
class SudokuCNN(nn.Module):
    def __init__(self, in_ch=10, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(X[0].size(0), hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, 9, 1)  # 각 칸 9클래스 logits
        )

    def forward(self, x):
        logits = self.net(x)
        return logits

## conv2d  
작은 필터(예: 3×3)를  
전체 격자에 공유해서  
이웃 패턴을 인식  
주변 칸을 보면서 규칙을 학습하는 연산


In [108]:
def masked_ce_loss(logits, y, mask):
    B = y.size(0)
    # (B,C,H,W) -> (B*H*W, C)
    logits = logits.permute(0, 2, 3, 1).reshape(B*81, 9)
    y = y.reshape(B*81)
    mask = mask.reshape(B*81)

    logits_m = logits[mask]
    y_m = y[mask]

    # 빈칸이 하나도 없는 배치면(거의 없지만) 안전 처리
    if logits_m.numel() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    return F.cross_entropy(logits_m, y_m)

In [109]:
import time

device = torch.device("cpu")

model = SudokuCNN(in_ch=10, hidden=64).to(device)

model.load_state_dict(
    torch.load("data/sudoku_model.pt", map_location=device)
)
model.eval()

# opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# model.train()
# for epoch in range(1, 16):  # 일단 16epoch만
#     t0 = time.time()
#     total_loss = 0.0
#     total_cells = 0
#     correct_cells = 0

#     for X, y, mask in loader:
#         X = X.to(device)
#         y = y.to(device)
#         mask = mask.to(device)

#         opt.zero_grad()
#         logits = model(X)  # (B, 9, 9, 9) (B,C,H,W)

#         loss = masked_ce_loss(logits, y, mask)
#         loss.backward()
#         opt.step()

#         total_loss += loss.item()

#         # 빈칸 cell accuracy (모니터링용)
#         with torch.no_grad():
#             pred = logits.argmax(dim=1)  # (B, 9, 9)
#             m = mask
#             total_cells += m.sum().item()
#             correct_cells += ((pred == y) & m).sum().item()

#     dt = time.time() - t0
#     acc = (correct_cells / total_cells) if total_cells > 0 else 0.0
#     print(f"epoch {epoch} | loss {total_loss/len(loader):.4f} | blank_acc {acc:.4f} | time {dt:.1f}s")


SudokuCNN(
  (net): Sequential(
    (0): Conv2d(10, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(64, 9, kernel_size=(1, 1), stride=(1, 1))
  )
)

In [110]:
import numpy as np
import torch
import copy

solution = sudoku.construct_puzzle_solution()
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)

quiz = np.array(puzzle, dtype=np.int64)  # (9,9)
mask = (quiz == 0)

# one-hot (10,9,9)
X = np.zeros((10, 9, 9), dtype=np.float32)
for d in range(10):
    X[d] = (quiz == d)

# torch 텐서: (1,10,9,9)
a = torch.from_numpy(X).unsqueeze(0)  # add batch dim

model.eval()
with torch.no_grad():
    logits = model(a)                 # (1,9,9,9) = (B,C,H,W)
    pred = logits.argmax(dim=1)[0]    # (9,9) 0~8

pred_1to9 = (pred + 1).cpu().numpy()  # (9,9)

filled = quiz.copy()
filled[mask] = pred_1to9[mask]

print("QUIZ:")
print(quiz)
print("\nFILLED:")
print(filled)


QUIZ:
[[7 0 0 0 0 0 3 0 0]
 [9 0 0 8 0 0 0 5 0]
 [0 0 4 1 2 6 8 0 0]
 [1 0 2 4 0 0 0 3 0]
 [4 0 0 0 0 1 0 0 6]
 [0 7 8 0 0 0 5 0 0]
 [0 4 0 9 5 0 0 1 0]
 [6 0 3 0 0 4 2 0 0]
 [0 0 5 2 6 0 0 7 9]]

FILLED:
[[7 1 8 9 5 4 3 2 6]
 [9 1 6 8 3 4 1 5 4]
 [3 8 4 1 2 6 8 9 1]
 [1 9 2 4 1 9 1 3 1]
 [4 9 6 5 8 1 4 2 6]
 [9 7 8 6 4 3 5 9 7]
 [2 4 1 9 5 3 6 1 5]
 [6 2 3 1 7 4 2 6 8]
 [8 8 5 2 6 3 4 7 9]]


In [111]:
import numpy as np
import torch

# 1. 보드 <-> 모델 입력 변환
def board_to_X(board: np.ndarray) -> torch.Tensor:
    """
    board: (9,9) int, 0=blank, 1~9=digits
    return: (1,10,9,9) float32 one-hot (channel 0 = blank)
    """
    X = np.zeros((10, 9, 9), dtype=np.float32)
    for d in range(10):
        X[d] = (board == d).astype(np.float32)
    return torch.from_numpy(X).unsqueeze(0)  # (1,10,9,9)


# 2. 스도쿠 제약 기반 후보 마스크
def valid_candidates(board: np.ndarray, r: int, c: int) -> np.ndarray:
    """
    return: (9,) bool mask for digits 1..9 that are valid at (r,c)
    """
    if board[r, c] != 0:
        return np.zeros(9, dtype=bool)

    used = set(board[r, :]) | set(board[:, c])
    br, bc = (r // 3) * 3, (c // 3) * 3
    used |= set(board[br:br + 3, bc:bc + 3].reshape(-1))
    used.discard(0)

    mask = np.ones(9, dtype=bool)
    for v in used:
        if 1 <= v <= 9:
            mask[v - 1] = False
    return mask


def build_valid_mask(board: np.ndarray) -> np.ndarray:
    """
    return: (9,9,9) bool mask; mask[r,c,d] True면 digit(d+1) 가능
    """
    m = np.zeros((9, 9, 9), dtype=bool)
    for r in range(9):
        for c in range(9):
            if board[r, c] == 0:
                m[r, c] = valid_candidates(board, r, c)
    return m


# 3. 모델 출력 형태 통일 (여기 중요)
def logits_to_9x9x9(logits: torch.Tensor) -> torch.Tensor:
    """
    모델이 어떤 형태로 내놓든 (1,9,9,9)로 통일.
    지원:
      - (B, 9, 9, 9)
      - (B, 81, 9)
      - (B, 9, 81)  (가끔)
    """
    if logits.dim() == 4 and logits.shape[-1] == 9:
        # (B,9,9,9)
        return logits
    if logits.dim() == 3 and logits.shape[1] == 81 and logits.shape[2] == 9:
        # (B,81,9) -> (B,9,9,9)
        return logits.view(logits.shape[0], 9, 9, 9)
    if logits.dim() == 3 and logits.shape[1] == 9 and logits.shape[2] == 81:
        # (B,9,81) -> (B,9,9,9)
        return logits.permute(0, 2, 1).contiguous().view(logits.shape[0], 9, 9, 9)
    raise ValueError(f"Unsupported logits shape: {tuple(logits.shape)}")


# 4. Autoregressive / Iterative Greedy Solver
@torch.no_grad()
def solve_iterative_greedy(
    model,
    board: np.ndarray,
    device: str = None,
    max_steps: int = 200,
    temperature: float = 1.0,
    return_trace: bool = False,
):
    """
    핵심 아이디어:
      1) 모델이 전체 칸 확률 예측
      2) 빈 칸 중 '가장 확신 높은' (r,c,값) 1개 선택
      3) 그 값 확정 후 보드 업데이트
      4) 반복

    - constraint masking 포함: 불가능 숫자는 확률 0 처리
    """
    model.eval()
    bd = board.copy().astype(np.int64)

    if device is None:
        # 모델 파라미터가 있는 device로 자동 추정
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
    else:
        device = torch.device(device)

    trace = []

    for step in range(max_steps):
        blanks = np.argwhere(bd == 0)
        if len(blanks) == 0:
            return (bd, trace) if return_trace else bd  # solved

        # (1,10,9,9)
        X = board_to_X(bd).to(device)

        logits = model(X)
        logits = logits_to_9x9x9(logits)  # (1,9,9,9)
        logits = logits[0]  # (9,9,9)

        # temperature
        if temperature != 1.0:
            logits = logits / float(temperature)

        # constraint mask: 불가능 숫자 -inf 처리
        valid = build_valid_mask(bd)  # (9,9,9) bool (numpy)
        valid_t = torch.from_numpy(valid).to(device)

        # 빈칸이 아닌 곳은 선택 대상에서 제외하려고, 일단 logits는 마스크만 적용
        masked_logits = logits.clone()
        masked_logits[~valid_t] = -1e9  # 불가능 digit 제거

        probs = torch.softmax(masked_logits, dim=-1)  # (9,9,9)

        # 빈칸 위치만 평가해서 "가장 확신 높은" 채우기 선택
        best_r, best_c, best_d = None, None, None
        best_p = -1.0

        for (r, c) in blanks:
            # 해당 칸에서 가장 높은 후보
            pvals = probs[r, c]  # (9,)
            p_max, d_idx = torch.max(pvals, dim=-1)
            p_max = float(p_max.item())
            d_idx = int(d_idx.item())  # 0..8

            # 만약 후보가 전부 막혔으면(=0), 모순 상태
            if p_max <= 0.0:
                # 모순: 해결 실패
                return (None, trace) if return_trace else None

            if p_max > best_p:
                best_p = p_max
                best_r, best_c, best_d = int(r), int(c), d_idx + 1  # digit 1..9

        # 선택한 값 확정
        bd[best_r, best_c] = best_d
        if return_trace:
            trace.append((best_r, best_c, best_d, best_p))

    # max_steps 초과: 아직 미해결
    return (None, trace) if return_trace else None


# 5. 사용
board = np.array(quiz, dtype=np.int64)  # (9,9), 0=blank
solved = solve_iterative_greedy(model, board, device="cpu", max_steps=300)
if solved is None:
    print("실패(모순 또는 max_steps 초과)")
else:
    print(solved)


[[7 8 1 5 4 9 3 6 2]
 [9 2 6 8 3 7 1 5 4]
 [3 5 4 1 2 6 8 9 7]
 [1 6 2 4 7 5 9 3 8]
 [4 3 9 1 8 1 7 2 6]
 [1 7 8 6 9 2 5 4 1]
 [2 4 7 9 5 8 6 1 3]
 [6 9 3 7 1 4 2 8 5]
 [8 1 5 2 6 3 4 7 9]]
